# Sparse Matrix Block Encoding

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389.
Copyright Google LLC. SPDX-License-Identifier: Apache-2.0.

In [ ]:
from collections import Counter

import torch
from IPython.display import Markdown, SVG, display
from sympy import Symbol

from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum import Circuit
from flagquantum.compiler._component_trace.call_graph import modeled_call_graph
from flagquantum.compiler._component_trace.resource_model import estimate_component_resources
from flagquantum.core.ir.program.register_types import opaque_register_type
from flagquantum.core.ir.program.registers import Register
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import (
    export_modeled_call_graph_dot, format_call_graph_markdown, render_call_graph_svg,
)
from flagquantum.simulation.unitary import get_unitary

def signature(width):
    return Signature((
        Register("system", opaque_register_type(width)),
        Register("ancilla", opaque_register_type(width + 1)),
    ))

## Sparse-access matrix

Row and column oracles map a valid ordinal to an entry index. The entry
oracle rotates a flag qubit so its zero amplitude holds the corresponding
matrix entry. For equal row and column sparsity $s$, the normalized block
is $A/s$. It uses an $n$-qubit system, an $n$-qubit ancillary ordinal and
one additional flag. The ordinal and system words use big-endian order.

Prepare a uniform ordinal, apply the column and entry oracles, swap the two
index words, apply the row adjoint and unprepare the ordinal. The row and
column oracles own their scratch only during their calls. The block
can reuse the same temporary allocation across sequential stages.

Pad irregular dimensions to a power of two, and include zero entries in
the oracle lists when rows or columns need equal lengths. The symbolic
instance provides shape and normalization metadata. Physical binding uses
concrete widths.

[Lecture Notes on Quantum Algorithms for Scientific Computation](https://arxiv.org/abs/2201.08309)
describes this oracle construction.

In [ ]:
from flagquantum.algorithms.primitives.sparse_matrix_block_encoding import SparseMatrixBlockEncoding
from flagquantum.algorithms.primitives.sparse_matrix_oracles import (
    ExplicitEntryOracle, SymmetricBandedRowColumnOracle,
    TopLeftRowColumnOracle, UniformEntryOracle,
)

### Example Instances

In [ ]:
top_left = TopLeftRowColumnOracle(2)
uniform = SparseMatrixBlockEncoding(top_left, top_left, UniformEntryOracle(2, 0.3))
assert uniform.alpha == 4

In [ ]:
n = Symbol("n", positive=True, integer=True)
symbolic_signature = signature(n)
symbolic_normalization = 2**n
print("Symbolic normalization", symbolic_normalization)

In [ ]:
index = TopLeftRowColumnOracle(1)
data = ((0.0, 0.25), (1 / 3, 0.467))
explicit = SparseMatrixBlockEncoding(index, index, ExplicitEntryOracle(1, data, 10))
print("Explicit entry precision", explicit.entry_oracle.entry_bitsize)

In [ ]:
band = SymmetricBandedRowColumnOracle(3, bandsize=1)
banded = SparseMatrixBlockEncoding(band, band, UniformEntryOracle(3, 0.3))
assert banded.alpha == 3

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([
    ("Uniform top-left block", signature(uniform.system_bitsize), {}),
    ("Symbolic top-left block", symbolic_signature, {}),
    ("Explicit entry block", signature(explicit.system_bitsize), {}),
    ("Symmetric banded block", signature(banded.system_bitsize), {}),
])))

### Call Graph

In [ ]:
model = uniform.resource_model()
graph = modeled_call_graph(model, max_depth=1)
display(SVG(render_call_graph_svg(export_modeled_call_graph_dot(model, max_depth=1))))
display(Markdown(format_call_graph_markdown(
    graph.root, graph.direct_calls, graph.leaf_counts, title=lambda node: node.name,
)))

### Verify the uniform block through append

Project onto the all-zero flag and ordinal to recover the encoded 4 × 4
matrix. The concrete uniform case has no temporary work. Compare its
primitive emission with the independent source model.

In [ ]:
binding = dict(system=(0, 1), ancilla=(2, 3, 4))
circuit = Circuit(5, dtype=torch.complex128)
circuit.append(uniform.make_on(**binding))
matrix = get_unitary(circuit)
selected = torch.arange(4) * 8
projected = uniform.alpha * matrix[selected][:, selected]
torch.testing.assert_close(projected, torch.full((4, 4), 0.3, dtype=torch.complex128),
                           atol=2e-12, rtol=0)
print("Emitted primitives", dict(Counter(gate.name for gate in circuit.to_ir().instructions)))
print("Source resource model", estimate_component_resources(model))
circuit.append(uniform.make_on(**binding, adjoint=True))
torch.testing.assert_close(get_unitary(circuit), torch.eye(32, dtype=torch.complex128),
                           atol=2e-12, rtol=0)